# ปฏิบัติการที่ 9: การประมวลผลข้อมูล
## สรุปโค้ดพร้อมคำอธิบายละเอียด T1–C5
---

## 📦 Library ที่ต้องติดตั้ง
รันเซลล์นี้ก่อนเริ่มใช้งาน

In [ ]:
# ติดตั้ง library ที่จำเป็น
# !pip install pythainlp pillow opencv-python librosa soundfile matplotlib

---
# หมวด A: Text Processing
---

## T1: นับคำและความถี่ (Word Frequency)

**เป้าหมาย:** รับข้อความ → ตัดคำ → นับความถี่ → แสดง 10 อันดับแรก

**Flow:**
```
ข้อความดิบ → re.sub() ตัดเครื่องหมาย → word_tokenize() → dict นับ → sorted() top10
```

In [ ]:
import re
from pythainlp.tokenize import word_tokenize

text = input('ใส่ข้อความ: ')

# ขั้นที่ 1: ตัดเครื่องหมายวรรคตอนออก
# [^ก-๙a-zA-Z0-9\s] = ยกเว้นไทย อังกฤษ เลข ช่องว่าง → ลบทิ้งทั้งหมด
text = re.sub(r'[^ก-๙a-zA-Z0-9\s]', '', text)

# ขั้นที่ 2: ตัดคำภาษาไทย
# .lower() แปลงอังกฤษเป็นตัวเล็กก่อน เพื่อไม่ให้ 'Hello' กับ 'hello' นับแยกกัน
# engine='newmm' คือ algorithm ตัดคำที่แม่นยำที่สุดของ pythainlp
words = word_tokenize(text.lower(), engine='newmm')

# ขั้นที่ 3: กรองช่องว่างออก
# w.strip() จะคืน '' (empty) ถ้าเป็นช่องว่าง → if กรองออก
words = [w for w in words if w.strip()]

# ขั้นที่ 4: นับความถี่ด้วย dict
# counter.get(word, 0) = ถ้าไม่มี key นี้ให้ใช้ 0 แทน
counter = {}
for word in words:
    counter[word] = counter.get(word, 0) + 1

# ขั้นที่ 5: เรียงลำดับและแสดงผล
# counter.items() = [(คำ, จำนวน), ...]
# key=lambda x: x[1] = เรียงตามจำนวน (ตัวที่ 2 ของ tuple)
# reverse=True = มากไปน้อย
# [:10] = เอาแค่ 10 อันดับแรก
top10 = sorted(counter.items(), key=lambda x: x[1], reverse=True)[:10]

print(f"\nจำนวนคำทั้งหมด: {len(words)}")
print("\n10 คำที่พบบ่อยสุด:")
for word, count in top10:
    print(f"  {word}: {count}")

## T2: ทำความสะอาดข้อความ (Text Cleaning)

**เป้าหมาย:** ลบช่องว่างซ้ำ → lowercase อังกฤษ → ลบอีโมจิ/พิเศษ

**Flow:**
```
ข้อความดิบ → re.sub() 3 รอบ → พิมพ์ก่อน/หลัง
```

In [ ]:
import re

text = input('ใส่ข้อความ: ')
print(f"ก่อน: {text}")

# ขั้นที่ 1: ลบช่องว่างซ้ำ
# r' +' = ช่องว่าง 1 ตัวขึ้นไป → แทนด้วยช่องว่างตัวเดียว
# .strip() = ตัดช่องว่างหัว-ท้ายออก
text = re.sub(r' +', ' ', text).strip()

# ขั้นที่ 2: แปลงอังกฤษเป็น lowercase
# [a-zA-Z]+ = หาคำอังกฤษทุกคำ
# lambda x: x.group().lower() = เอาคำที่เจอมาแปลงเป็นตัวเล็ก
# ภาษาไทยไม่โดนแตะเลย
text = re.sub(r'[a-zA-Z]+', lambda x: x.group().lower(), text)

# ขั้นที่ 3: ลบอีโมจิและอักขระพิเศษ
# เหมือน T1 เลย ลบทุกอย่างที่ไม่ใช่ไทย อังกฤษ เลข ช่องว่าง
text = re.sub(r'[^ก-๙a-zA-Z0-9\s]', '', text)

print(f"หลัง: {text}")

## T3: สรุปความยาวประโยค

**เป้าหมาย:** แบ่งประโยค → นับคำแต่ละประโยค → หาประโยคยาวสุดและค่าเฉลี่ย

**Flow:**
```
ข้อความ → re.split([.?!]) → นับคำแต่ละประโยค → max() + เฉลี่ย → พิมพ์
```

In [ ]:
import re

text = input('ใส่ข้อความ: ')

# ขั้นที่ 1: แบ่งประโยคโดยใช้ . ? ! เป็นตัวคั่น
# re.split() แบ่งข้อความตาม pattern
# [.?!] = จุด หรือ ? หรือ !
sentences = re.split(r'[.?!]', text)

# ขั้นที่ 2: กรองประโยคว่างออก
# s.strip() ตัดช่องว่างหัว-ท้าย ถ้าว่างก็กรองออก
sentences = [s.strip() for s in sentences if s.strip()]

# ขั้นที่ 3: นับคำแต่ละประโยค
# s.split() แบ่งคำด้วยช่องว่าง แล้ว len() นับจำนวน
word_counts = [len(s.split()) for s in sentences]

# ขั้นที่ 4: หาประโยคที่ยาวสุด
# max() หาค่ามากสุด
# .index() หาตำแหน่งของค่านั้น
longest_idx = word_counts.index(max(word_counts))

# ขั้นที่ 5: แสดงผล
print(f"\nจำนวนประโยค    : {len(sentences)}")
print(f"เฉลี่ยคำ/ประโยค : {sum(word_counts)/len(word_counts):.2f}")
print(f"ประโยคที่ยาวสุด : {sentences[longest_idx]}")
print(f"จำนวนคำ        : {word_counts[longest_idx]}")

## T4: ตรวจจับสแปม (Spam Filter)

**เป้าหมาย:** ให้คะแนนข้อความตาม keyword, ลิงก์, ตัวซ้ำ → ตัดสินระดับ

**เกณฑ์คะแนน:**
- keyword น่าสงสัย = +2 ต่อคำ
- มีลิงก์ http/https = +3
- ตัวซ้ำเกิน 4 ตัว = +2
- 0 คะแนน = ปกติ / 1-3 = น่าสงสัย / 4+ = สแปม

In [ ]:
import re

# คำที่น่าสงสัย
keywords = ['ฟรี', 'ด่วน', 'คลิก', 'โปรโมชั่น', 'เงินคืน', 'รับทันที']

text  = input('ใส่ข้อความ: ')
score = 0

# ขั้นที่ 1: เช็ค keyword
# loop ทุกคำใน keywords แล้วเช็คว่ามีในข้อความไหม
for word in keywords:
    if word in text:
        score += 2
        print(f"  พบคำ '{word}' +2")

# ขั้นที่ 2: เช็คลิงก์
# https? = http หรือ https (? = มีหรือไม่มี s ก็ได้)
# :// = ตามด้วย ://
if re.search(r'https?://', text):
    score += 3
    print("  พบลิงก์ +3")

# ขั้นที่ 3: เช็คตัวซ้ำ
# (.) = จับ 1 ตัวอักษร
# \1  = ต้องซ้ำตัวเดิม
# {4,} = ซ้ำ 4 ครั้งขึ้นไป
# รวมกัน = ตัวอักษรซ้ำกัน 5 ตัวขึ้นไปติดกัน
if re.search(r'(.)\1{4,}', text):
    score += 2
    print("  พบตัวซ้ำ +2")

# ขั้นที่ 4: ตัดสิน
level = "ปกติ" if score == 0 else "น่าสงสัย" if score <= 3 else "สแปม"
print(f"\nคะแนน: {score} → {level}")

## T5: Mini Search Engine

**เป้าหมาย:** สร้าง index จากไฟล์ .txt → รับคำค้น → แสดงไฟล์ที่เกี่ยวข้อง

**โครงสร้าง index:**
```python
{
    'python': {'file1.txt': 3, 'file2.txt': 1},
    'การ':    {'file1.txt': 2}
}
```

In [ ]:
import os

def build_index(folder):
    """สร้าง index จากทุกไฟล์ .txt ในโฟลเดอร์"""
    index = {}
    for filename in os.listdir(folder):
        if filename.endswith('.txt'):
            path  = os.path.join(folder, filename)
            # อ่านไฟล์และตัดคำด้วยช่องว่าง
            words = open(path, encoding='utf-8').read().lower().split()
            for word in words:
                # ถ้าคำนี้ยังไม่มีใน index ให้สร้าง dict ใหม่
                if word not in index:
                    index[word] = {}
                # นับจำนวนครั้งที่พบในแต่ละไฟล์
                index[word][filename] = index[word].get(filename, 0) + 1
    return index

def search(index, query):
    """ค้นหาคำและแสดงผล sorted ตามจำนวนครั้ง"""
    results = index.get(query.lower(), {})
    if not results:
        print("ไม่พบคำนี้ในไฟล์ใดเลย")
        return
    # เรียงตามจำนวนครั้งที่พบ มากไปน้อย
    sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)
    print(f"\nผลการค้นหา '{query}':")
    for filename, count in sorted_results:
        print(f"  {filename}: พบ {count} ครั้ง")

# รัน
folder = 'texts'   # โฟลเดอร์ที่เก็บไฟล์ .txt
index  = build_index(folder)
query  = input("ค้นหาคำ: ")
search(index, query)

---
# หมวด B: Image Processing (Pillow)
---

## B1: แปลงภาพ + สถิติสี

**เป้าหมาย:** แปลงภาพเป็น grayscale → คำนวณ mean และ std

**Flow:**
```
Image.open() → .convert('L') → np.array() → .mean() / .std() → .save()
```

In [ ]:
from PIL import Image
import numpy as np

# ขั้นที่ 1: เปิดภาพ
img = Image.open('photo.jpg')

# ขั้นที่ 2: แปลงเป็น grayscale
# 'L' = Luminance = ความสว่าง (0=ดำ, 255=ขาว)
# Pillow ใช้สูตร: 0.299R + 0.587G + 0.114B (ถ่วงน้ำหนักตามสายตามนุษย์)
gray = img.convert('L')

# ขั้นที่ 3: แปลงเป็น array ตัวเลขเพื่อคำนวณ
# np.array() เปลี่ยนภาพเป็นตาราง 2D ของตัวเลข 0-255
arr  = np.array(gray)
mean = arr.mean()   # ความสว่างเฉลี่ย
std  = arr.std()    # ส่วนเบี่ยงเบน (ยิ่งสูง = คอนทราสต์เยอะ)

# ขั้นที่ 4: บันทึกและแสดงผล
gray.save('photo_gray.jpg')
print(f"Mean brightness : {mean:.2f}")
print(f"Std deviation   : {std:.2f}")

## B1 (แบบ manual): คำนวณ grayscale ด้วย (R+G+B)//3

แบบนี้ไม่ใช้ `.convert('L')` แต่คำนวณทีละ pixel เอง

In [ ]:
from PIL import Image
import numpy as np

img    = Image.open('photo.jpg')
pixels = img.load()      # pixel access object อ่าน/เขียนแต่ละ pixel ได้
w, h   = img.size        # img.size คืน (width, height)

# สร้างภาพใหม่เปล่าๆ โหมด 'L' = grayscale
gray_img    = Image.new('L', (w, h))
gray_pixels = gray_img.load()

# วน loop ทุก pixel
# y = แถว (บน→ล่าง), x = คอลัมน์ (ซ้าย→ขวา)
for y in range(h):
    for x in range(w):
        r, g, b = pixels[x, y]       # อ่านสี RGB
        gray    = (r + g + b) // 3   # เฉลี่ย 3 ช่อง, // ปัดลงให้เป็น int
        gray_pixels[x, y] = gray     # เขียนลงภาพใหม่

arr  = np.array(gray_img)
mean = arr.mean()
std  = arr.std()

gray_img.save('gray_manual.jpg')
print(f"Mean : {mean:.2f}")
print(f"Std  : {std:.2f}")

## B2: ย่อ/ขยายภาพแบบรักษาสัดส่วน

**เป้าหมาย:** ปรับความกว้างตามที่กำหนด โดยความสูงปรับตามสัดส่วนอัตโนมัติ

**หลักการ:** ratio = ความกว้างใหม่ / ความกว้างเดิม → คูณความสูงด้วย ratio

In [ ]:
from PIL import Image

img = Image.open('photo.jpg')
print(f"ขนาดเดิม: {img.size}")   # (width, height)

target_w = 512

# ratio = อยากได้กี่เปอร์เซ็นต์ของความกว้างเดิม
# เช่น 512/1024 = 0.5 = อยากได้ 50%
ratio = target_w / img.width

# ความสูงต้องลดลงในอัตราส่วนเดียวกัน
# int() ปัดลงเพราะ pixel ต้องเป็นจำนวนเต็ม
new_h = int(img.height * ratio)

# .resize() รับ tuple (width, height) เท่านั้น!
# Image.LANCZOS = algorithm ที่ให้ภาพคมชัดดีที่สุด
resized = img.resize((target_w, new_h), Image.LANCZOS)
resized.save('photo_resized.jpg')
print(f"ขนาดใหม่: {resized.size}")

## B3: ตรวจจับขอบ (Edge Detection)

**เป้าหมาย:** หาเส้นขอบของวัตถุในภาพ

**2 วิธี:** Pillow (ง่าย) หรือ OpenCV + Canny (นิยมกว่า)

In [ ]:
# วิธีที่ 1: Pillow
from PIL import Image, ImageFilter

img   = Image.open('photo.jpg')
gray  = img.convert('L')                        # ต้องแปลง grayscale ก่อน
edges = gray.filter(ImageFilter.FIND_EDGES)     # หาเส้นขอบ
edges.save('edges_pillow.jpg')
print("Pillow: บันทึกเรียบร้อย")

In [ ]:
# วิธีที่ 2: OpenCV + Canny (นิยมกว่า)
import cv2

img     = cv2.imread('photo.jpg')                       # OpenCV อ่านเป็น BGR!
gray_cv = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)         # แปลงเป็น grayscale

# Canny(ภาพ, threshold1, threshold2)
# threshold1, threshold2 = ความไว ถ้าไม่บอกใช้ 100, 200
edges   = cv2.Canny(gray_cv, 100, 200)
cv2.imwrite('edges_canny.jpg', edges)
print("Canny: บันทึกเรียบร้อย")

## B4: เบลอเฉพาะพื้นที่ (Selective Blur)

**เป้าหมาย:** เบลอเฉพาะพื้นที่ในกรอบ (x1, y1, x2, y2)

**Flow:**
```
.crop() ตัดออก → .filter() เบลอ → .paste() วางกลับ → .save()
```

**ทำไมต้อง paste?** เพราะ `.crop()` คืน object ใหม่แยกต่างหาก ไม่ได้แก้ภาพต้นฉบับ ต้องวางกลับเองเสมอ

In [ ]:
from PIL import Image, ImageFilter

img = Image.open('photo.jpg')

# พิกัดกรอบ (x1, y1, x2, y2)
# x1, y1 = มุมบนซ้าย
# x2, y2 = มุมล่างขวา
x1, y1, x2, y2 = 100, 100, 300, 300

# ขั้นที่ 1: ตัดส่วนที่จะเบลอออกมา
region = img.crop((x1, y1, x2, y2))

# ขั้นที่ 2: เบลอส่วนนั้น
# radius ยิ่งมาก ยิ่งเบลอมาก (2=นิดหน่อย, 15=ปานกลาง, 30=มาก)
blurred_region = region.filter(ImageFilter.GaussianBlur(radius=15))

# ขั้นที่ 3: วางกลับคืนตำแหน่งเดิม
# paste(ภาพที่จะวาง, (x, y) ตำแหน่งมุมบนซ้าย)
img.paste(blurred_region, (x1, y1))

img.save('photo_blurred.jpg')
print("บันทึกเรียบร้อย!")

---
# หมวด C: Audio Processing (librosa)
---

**สิ่งสำคัญที่สุด:** ทุกข้อเริ่มด้วยบรรทัดนี้เสมอ
```python
y, sr = librosa.load('audio.wav', sr=None)
# y  = array ตัวเลขเสียง [-1, 1]
# sr = sample rate (จำนวน sample ต่อวินาที)
# sr=None = ใช้ sample rate ของไฟล์จริง
```

## C1: ดูข้อมูลพื้นฐานไฟล์เสียง

**เป้าหมาย:** แสดงความยาว, sample rate, mono/stereo

In [ ]:
import librosa

# ใช้ไฟล์ตัวอย่างของ librosa (ไม่ต้องมีไฟล์เอง)
path   = librosa.example('trumpet')
y, sr  = librosa.load(path, sr=None)

# ความยาว = จำนวน sample ทั้งหมด / sample ต่อวินาที
length   = len(y) / sr

# ndim = จำนวนมิติ
# 1 มิติ = mono (ช่องเดียว)
# 2 มิติ = stereo (2 ช่อง)
channels = 'mono' if y.ndim == 1 else 'stereo'

print(f"Length   : {length:.2f} วินาที")
print(f"Sr       : {sr} Hz")
print(f"Channels : {channels}")

## C2: ปรับความดัง + Normalize

**เป้าหมาย:** ทำให้ peak ของเสียงอยู่ที่ 1.0 (ดังสุดเท่าที่ไม่แตก)

**หลักการ:** หารทุกค่าด้วยค่า peak → peak ใหม่จะเป็น 1.0 เสมอ

In [ ]:
import librosa
import numpy as np
import soundfile as sf

path  = librosa.example('trumpet')
y, sr = librosa.load(path, sr=None)

# np.abs() เอาค่าสัมบูรณ์ (ลบเป็นบวก)
# np.max() หาค่าสูงสุด
# รวมกัน = หา peak ของสัญญาณเสียง
peak = np.max(np.abs(y))
print(f"Peak ก่อน : {peak:.4f}")

# หารทุกค่าด้วย peak = normalize
# ค่าสูงสุดใหม่จะเป็น 1.0 เสมอ
y_norm = y / peak
print(f"Peak หลัง : {np.max(np.abs(y_norm)):.4f}")  # ควรได้ 1.0000

# soundfile.write() บันทึกไฟล์เสียง
sf.write('output_normalized.wav', y_norm, sr)
print("บันทึกเรียบร้อย!")

## C3: ตัดเงียบหัว-ท้าย (Trim Silence)

**เป้าหมาย:** ตัดช่วงเสียงเงียบที่หัวและท้ายไฟล์ออก

In [ ]:
import librosa
import soundfile as sf

path  = librosa.example('trumpet')
y, sr = librosa.load(path, sr=None)

old_length = len(y) / sr
print(f"ความยาวก่อน : {old_length:.2f} วินาที")

# librosa.effects.trim() คืน tuple 2 ค่า
# y_trim = array เสียงที่ตัดแล้ว
# _      = index ที่ตัด (ไม่ได้ใช้ ทิ้งด้วย _)
y_trim, _ = librosa.effects.trim(y)

new_length = len(y_trim) / sr
print(f"ความยาวหลัง : {new_length:.2f} วินาที")

sf.write('trimmed.wav', y_trim, sr)
print("บันทึกเรียบร้อย!")

## C4: แยกช่วงเสียงพูด/ไม่พูด (Voice Activity Detection)

**เป้าหมาย:** แบ่งเสียงเป็น frame → วัด energy → ถ้าเกิน threshold = มีเสียง

**หลักการ:** energy = ผลรวมของค่ายกกำลังสอง → เสียงดังมี energy สูง

In [ ]:
import librosa
import numpy as np

path  = librosa.example('trumpet')
y, sr = librosa.load(path, sr=None)

# ขนาด frame = 20ms
# sr * 0.02 = จำนวน sample ใน 20ms
# int() ปัดลงให้เป็นจำนวนเต็ม
frame_length = int(sr * 0.02)
threshold    = 0.001   # ปรับได้ ถ้าเยอะเกินเพิ่ม ถ้าน้อยเกินลด

print("ช่วงเวลาที่มีเสียง:")
# วน loop ทีละ frame
# range(0, len(y), frame_length) = 0, frame_length, frame_length*2, ...
for i in range(0, len(y), frame_length):
    frame  = y[i : i + frame_length]    # ตัด frame ออกมา
    energy = np.sum(frame ** 2)         # คำนวณ energy
    if energy > threshold:
        start = i / sr                  # แปลง sample index → วินาที
        end   = (i + frame_length) / sr
        print(f"  {start:.2f}s - {end:.2f}s")

## C5: ทำ Spectrogram แล้วบันทึกเป็นรูป

**เป้าหมาย:** แปลงเสียงเป็นภาพ spectrogram (แกน x=เวลา, y=ความถี่)

**Flow:**
```
load → stft → amplitude_to_db → specshow → savefig
```

In [ ]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

path  = librosa.example('trumpet')
y, sr = librosa.load(path, sr=None)

# Short-Time Fourier Transform
# แปลงเสียง (time domain) → ความถี่ (frequency domain)
# คืนค่าเป็น complex number
S = librosa.stft(y)

# np.abs() เอาค่าสัมบูรณ์ของ complex number = amplitude
# amplitude_to_db() แปลงเป็น dB ให้อ่านง่ายขึ้น
S_db = librosa.amplitude_to_db(np.abs(S))

# วาดกราฟ
plt.figure(figsize=(10, 4))

# specshow() วาด spectrogram
# x_axis='time' = แกน x เป็นเวลา
# y_axis='hz'   = แกน y เป็นความถี่ Hz
librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='hz')

plt.colorbar(format='%+2.0f dB')   # แสดง scale สี
plt.title('Spectrogram')
plt.tight_layout()
plt.savefig('spectrogram.png')      # บันทึกเป็นรูป
plt.show()
print("บันทึก spectrogram.png เรียบร้อย!")

---
## สรุป Pattern ที่ต้องจำ

| ข้อ | function หลัก |
|---|---|
| T1 | `word_tokenize()` + `dict.get()` + `sorted()` |
| T2 | `re.sub()` 3 รอบ |
| T3 | `re.split()` + `max()` |
| T4 | `in` + `re.search()` + คะแนน |
| T5 | `dict` ซ้อน `dict` + `sorted()` |
| B1 | `.convert('L')` → `np.array()` → `.mean()` `.std()` |
| B2 | `ratio = target/width` → `.resize((w, h))` |
| B3 | `.filter(FIND_EDGES)` หรือ `cv2.Canny()` |
| B4 | `.crop()` → `.filter()` → `.paste()` → `.save()` |
| C1 | `len(y)/sr` + `y.ndim` |
| C2 | `np.max(np.abs(y))` → หาร → `sf.write()` |
| C3 | `librosa.effects.trim(y)` คืน 2 ค่า |
| C4 | `frame` + `np.sum(frame**2)` + threshold |
| C5 | `stft` → `amplitude_to_db` → `specshow` → `savefig` |